In [ ]:
import json
import logging
from dataclasses import replace

import pandas as pd
from IPython.display import display

from elo_model.analysis import fingerprint
from elo_model.config import (
    NEURAL_ARTIFACT_DIRECTORY,
    NEURAL_TRAINING_DIRECTORY,
    FeatureConfig,
    PipelineConfig,
)
from elo_model.evaluation import rating_bands
from elo_model.feature_schema import FEATURE_NAMES
from elo_model.neural_training import TorchModelTrainer
from elo_model.pipeline import (
    analyze_dataset,
    export_sample,
    extract_dataset,
    feature_missingness,
    load_feature_metadata,
    manifest_rows,
    save_feature_dataset,
    split_summary,
)
from elo_model.predictor import RatingPredictor, export_bundle
from elo_model.split import PlayerSplitter
from elo_model.storage import write_json, write_parquet
from training_data import ChessComConnection, ModelDatasetBuilder
from training_data.preparation import PlayerPoolBuilder

settings = replace(
    PipelineConfig.expanded(),
    artifact_directory=NEURAL_ARTIFACT_DIRECTORY,
    training_directory=NEURAL_TRAINING_DIRECTORY,
)
allow_underfilled_bands = False

logging.basicConfig(
    level=settings.collection.log_level,
    format=settings.collection.log_format,
    force=True,
)

print(f"Model output: {settings.artifact_directory.resolve()}")
print(f"PyTorch MLP: {settings.neural_training.hidden_sizes}; device={settings.neural_training.device}")
print(f"Analysis: {settings.engine.time_seconds}s per position, MultiPV={settings.engine.multipv}")
print(f"Features: {len(FEATURE_NAMES)} (schema {settings.features.schema_version})")
print(f"Candidate pool target: {settings.candidate_games:,} unique games before selection")
print(f"Training quota target: {sum(quota.rows for quota in settings.stratification.quotas):,} player rows")
print(f"Collection months: {settings.collection.months}")
print(f"Discovery months: {settings.discovery_months}")
print(f"Discovery enabled: {settings.discovery_enabled}; extra seeds: {settings.discovery_seed_usernames}")
print(f"Stop discovery after {settings.collection.search_stagnation_expansions} expansions without quota improvement")
print(f"Shared raw archive cache: {settings.collection.archive_cache_directory}")
print(f"Username pool: {settings.usernames_path.resolve()}")
print(f"Persistent engine cache: {settings.engine_cache_directory.resolve()}")

C:\Users\mc678p\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model output: C:\Users\mc678p\OneDrive - AT&T Services, Inc\Documents\Tower Strategy & Roaming\Analysis\ELO Estimator\artifacts\rating_model_expanded
Analysis: 0.1s per position, MultiPV=1
Features: 93 (schema 2)
Candidate pool target: 60,000 unique games before selection
Training quota target: 20,000 player rows
Collection months: ((2026, 10), (2026, 9), (2026, 8), (2026, 7), (2026, 6), (2026, 5))
Discovery months: ((2026, 10),)
Discovery enabled: True; extra seeds: ('Hikaru', 'GothamChess')
Stop discovery after 500 expansions without quota improvement
Shared raw archive cache: data\archive_cache
Username pool: C:\Users\mc678p\OneDrive - AT&T Services, Inc\Documents\Tower Strategy & Roaming\Analysis\ELO Estimator\data\expanded\usernames.json
Persistent engine cache: C:\Users\mc678p\OneDrive - AT&T Services, Inc\Documents\Tower Strategy & Roaming\Analysis\ELO Estimator\data\analysis_cache


In [3]:
settings = replace(settings, target_games=2000)
print(f"Candidate target: {settings.candidate_games:,}")

Candidate target: 6,000


In [4]:
settings.data_directory.mkdir(parents=True, exist_ok=True)

collection_settings = {
    "months": [list(month) for month in settings.collection.months],
    "source_player_cap": settings.collection.max_games_per_player,
}

if settings.collection_settings_path.exists():
    previous_settings = json.loads(settings.collection_settings_path.read_text(encoding="utf-8"))
    if any(previous_settings.get(key) != value for key, value in collection_settings.items()):
        raise ValueError(
            "Collection months or source-player cap changed. Restore them or choose "
            "another data_directory; keep engine_cache_directory to reuse evaluations."
        )
elif settings.raw_games_path.exists():
    raise ValueError("Cached games have no collection provenance; choose another data_directory")

usernames = PlayerPoolBuilder(settings).build(progress=True)
collection_settings["usernames_hash"] = fingerprint(usernames)
collection_settings["usernames_count"] = len(usernames)
write_json(settings.collection_settings_path, collection_settings)

connection = ChessComConnection(settings.collection)
builder = ModelDatasetBuilder(connection)

print(f"Ready: {len(usernames):,} saved and discovered usernames")
print("Completed category searches are reused; adding seeds reruns discovery.")

Reusing blitz discovery: 3,000 usernames
Reusing rapid discovery: 3,000 usernames
Reusing bullet discovery: 3,000 usernames
Ready: 15,219 saved and discovered usernames
Completed category searches are reused; adding seeds reruns discovery.


In [5]:
logger = logging.getLogger("elo_model.notebook")
cached_games = None

if settings.raw_games_path.exists():
    cached_games = pd.read_parquet(settings.raw_games_path)
    print(f"Reusing {len(cached_games):,} previously collected games")

print(f"Collecting up to {settings.candidate_games:,} candidate games before selection")
games_df = builder.build(
    usernames,
    target_games=settings.candidate_games,
    initial_games=cached_games,
    progress=settings.training.progress,
    checkpoint=lambda frame: write_parquet(settings.raw_games_path, frame),
)

write_parquet(settings.raw_games_path, games_df)

if len(games_df) < settings.candidate_games:
    logger.warning(
        "Candidate pool contains %s/%s games; quotas may be underfilled. "
        "More usernames or archive months may improve coverage.",
        len(games_df), settings.candidate_games,
    )

print(f"Ready: {len(games_df):,} unique clocked candidate games")
print("Rejected games:", dict(builder.rejections))

Reusing 12,000 previously collected games
Ready: 6,000 unique clocked candidate games
Rejected games: {}


In [6]:
ratings = pd.concat(
    [games_df["white_rating"], games_df["black_rating"]],
    ignore_index=True,
)
categories = pd.concat([games_df["category"], games_df["category"]], ignore_index=True)

print("Collected game categories")
display(games_df["category"].value_counts().rename("games").to_frame())

print("Candidate player-side coverage by rating and time category")
display(pd.crosstab(
    pd.Series(rating_bands(ratings, expanded=True), name="rating_band"),
    categories.rename("category"),
    dropna=False,
))

print("Candidate rows are not training quotas; inspect the selected split next.")
display(games_df.head())

Collected game categories


,games
category,
blitz,3102
rapid,1610
bullet,1288


Candidate player-side coverage by rating and time category


category,blitz,bullet,rapid
rating_band,,,
<100,0,0,0
100-350,650,254,245
350-600,613,257,438
600-850,619,219,566
850-1100,922,255,451
1100-1350,754,262,286
1350-1600,513,229,405
1600-1850,273,166,300
1850-2100,445,250,205


Candidate rows are not training quotas; inspect the selected split next.


,game_id,pgn,white_username,black_username,white_rating,black_rating,category,termination,time_control
0,31220bd7a8ed80e6a9216eeeadb9a2b7000d5f5efdc5a5...,"[Event ""Live Chess""]\n[Site ""Chess.com""]\n[Dat...",Brucemiller123,RKaarav,515,518,rapid,checkmate,600
1,f5288678825c60cede8c7ce58590c2028bc9647eb20b18...,"[Event ""Live Chess""]\n[Site ""Chess.com""]\n[Dat...",Rkgty,RKaarav,526,501,rapid,checkmate,900+10
2,307423de0538032e28198f857234a39c6fdc8ecb38bb07...,"[Event ""Live Chess""]\n[Site ""Chess.com""]\n[Dat...",RKaarav,mmmmmmaaaatttttttttttt,358,393,blitz,timeout,180+2
3,60c766c41c9ad2225766b3927a844d5bf7f2f5f68f1604...,"[Event ""Live Chess""]\n[Site ""Chess.com""]\n[Dat...",RKaarav,valera_pvs,366,384,blitz,resignation,180+2
4,92b423034024c58cccd965c0b5e7aab93c766ee704dfb5...,"[Event ""Live Chess""]\n[Site ""Chess.com""]\n[Dat...",Xupa_cups,RKaarav,363,357,blitz,checkmate,180+2


In [7]:
from elo_model.selection import StratifiedGameSelector

coverage_approved = False
splitter = PlayerSplitter(
    fractions=settings.split_fractions,
    seed=settings.training.seed,
    strict_participants=settings.strict_participants,
)

split_candidates = splitter.assign(games_df)
selector = StratifiedGameSelector(settings.stratification, seed=settings.training.seed)
split_games = selector.select(split_candidates)
group_coverage = selector.partition_coverage(split_games)

selection_metadata = {
    "config": settings.stratification.model_dump(mode="json"),
    "seed": settings.training.seed,
    "candidate_games": len(games_df),
    "coverage": selector.coverage.to_dict("records"),
    "partition_coverage": group_coverage.to_dict("records"),
    "allow_underfilled_bands": allow_underfilled_bands,
}
write_json(settings.split_path.with_suffix(".selection.json"), selection_metadata)

if not selector.coverage.empty:
    with pd.option_context("display.max_rows", None):
        display(selector.coverage)
    if selector.coverage["shortfall"].any():
        logger.warning("Some training quotas are underfilled; inspect the selection coverage report.")

coverage = pd.DataFrame.from_dict(split_summary(split_games), orient="index")
display(coverage)
with pd.option_context("display.max_rows", None):
    display(group_coverage)

insufficient_rows = coverage["rows"] < settings.training.min_rows
insufficient_players = coverage["target_players"] < settings.min_players_per_partition
if insufficient_rows.any() or insufficient_players.any():
    raise ValueError("Selected split is too small; collect more candidates before running Stockfish.")

write_parquet(settings.split_path, split_games)
training_groups = group_coverage.loc[group_coverage["partition"] == "train"]
holdout_groups = group_coverage.loc[group_coverage["partition"] != "train"]
shortfalls = not selector.coverage.empty and selector.coverage["shortfall"].any()
sparse_training = (training_groups["players"] < settings.min_training_group_players).any()
sparse_holdouts = (holdout_groups["players"] < settings.min_holdout_group_players).any()

if sparse_holdouts:
    logger.warning(
        "Some validation/test groups have fewer than %s players; their metrics may be unreliable.",
        settings.min_holdout_group_players,
    )

needs_review = settings.min_training_group_players > 0 and (shortfalls or sparse_training)
allow_underfilled_bands = True
coverage_approved = not needs_review or allow_underfilled_bands
if not coverage_approved:
    raise ValueError(
        "Coverage needs review before Stockfish. Add discovery seeds or increase the candidate "
        "budget and rerun Cells 2-5. To accept documented shortfalls, set "
        "allow_underfilled_bands = True and rerun this cell."
    )

side_rows = manifest_rows(split_games)
side_rows["rating_band"] = rating_bands(side_rows["rating"], expanded=True)
display(pd.crosstab(side_rows["partition"], side_rows["rating_band"], dropna=False))
display(pd.crosstab(side_rows["partition"], side_rows["category"]))

,rating_band,category,requested_rows,available_rows,available_players,selected_rows,selected_players,shortfall
0,100-350,blitz,1000,495,270,442,270,558
1,100-350,rapid,500,173,91,146,90,354
2,100-350,bullet,500,174,97,153,96,347
3,350-600,blitz,1000,460,241,384,241,616
4,350-600,rapid,500,308,176,286,170,214
5,350-600,bullet,500,207,113,161,111,339
6,600-850,blitz,1000,444,257,419,257,581
7,600-850,rapid,500,407,218,322,218,178
8,600-850,bullet,500,144,79,120,79,380
9,850-1100,blitz,1000,717,351,601,351,399


2026-10-30 15:49:17,703 WARNING [MainThread] Some training quotas are underfilled; inspect the selection coverage report.


,rows,games,target_players
train,6924,4390,4137
validation,654,605,385
test,639,579,347


,partition,rating_band,category,rows,players
0,train,100-350,blitz,442,270
1,train,100-350,rapid,146,90
2,train,100-350,bullet,153,96
3,train,350-600,blitz,384,241
4,train,350-600,rapid,286,170
5,train,350-600,bullet,161,111
6,train,600-850,blitz,419,257
7,train,600-850,rapid,322,218
8,train,600-850,bullet,120,79
9,train,850-1100,blitz,601,351


2026-10-30 15:49:17,937 WARNING [MainThread] Some validation/test groups have fewer than 20 players; their metrics may be unreliable.


rating_band,<100,100-350,350-600,600-850,850-1100,1100-1350,1350-1600,1600-1850,1850-2100,2100-2350,2350-2600,>2600
partition,,,,,,,,,,,,
test,0,74,58,71,77,68,47,66,30,73,53,22
train,0,741,831,861,1005,782,697,466,568,516,457,0
validation,0,54,66,77,74,99,67,29,59,53,63,13


category,blitz,bullet,rapid
partition,,,
test,256,190,193
train,3622,1405,1897
validation,343,130,181


In [8]:
if not coverage_approved:
    raise ValueError("Review and approve split coverage in Cell 5 before engine analysis.")

engine_metadata = analyze_dataset(
    split_games,
    config=settings.engine,
    cache_directory=settings.engine_cache_directory,
    output=settings.analyzed_path,
    progress=settings.training.progress,
)

analyzed_games = pd.read_parquet(settings.analyzed_path)

print(f"Analysis references saved for {len(analyzed_games):,} games")
print("Rerunning this cell reuses complete cached evaluations")

Analyze games: 100%|██████████| 5574/5574 [05:27, cached=5574, new=0, fresh_s_per_game=unknown, fresh_remaining=0, fresh_ETA=00:00]


Analysis references saved for 5,574 games
Rerunning this cell reuses complete cached evaluations


In [ ]:
features_df = extract_dataset(
    analyzed_games,
    cache_directory=settings.engine_cache_directory,
    config=settings.features,
    progress=settings.training.progress,
    engine_metadata=engine_metadata,
)

save_feature_dataset(
    features_df,
    output=settings.features_path,
    engine_metadata=engine_metadata,
    config=settings.features,
)

missingness = pd.Series(feature_missingness(features_df), name="missing_fraction")

display(missingness.sort_values(ascending=False).to_frame())
print(f"Saved {len(features_df):,} player rows with {len(FEATURE_NAMES)} features")

Extract features:  67%|██████▋   | 3759/5574 [51:05:49<20:21,  1.49game/s]         

In [ ]:
feature_metadata = load_feature_metadata(settings.features_path)
training_rows = pd.read_parquet(settings.features_path)
training_feature_config = FeatureConfig(**feature_metadata["feature_config"])

trainer = TorchModelTrainer(settings.neural_training, training_feature_config)
result = trainer.fit(training_rows)

result.metadata.update(feature_metadata)
result.metadata["dataset_games"] = training_rows["game_id"].nunique()
result.metadata["split_fractions"] = list(settings.split_fractions)
result.metadata["strict_participants"] = settings.strict_participants
selection_path = settings.split_path.with_suffix(".selection.json")
if selection_path.exists():
    result.metadata["selection"] = json.loads(selection_path.read_text(encoding="utf-8"))

print("Neural training complete; best epoch selected on validation MAE only")

Train models: 100%|██████████| 1/1 [00:59<00:00, 59.85s/model]

Training complete; evaluation used untouched test rows after selection


In [ ]:
metric_rows = []

for model_name, report in result.report.items():
    for partition, estimators in report["partitions"].items():
        for estimator_name, metrics in estimators.items():
            metric_rows.append({
                "model": model_name,
                "partition": partition,
                "estimator": estimator_name,
                **metrics["overall"],
            })

metrics_df = pd.DataFrame(metric_rows)
display(metrics_df)

for model_name, report in result.report.items():
    print(f"{model_name}: best epoch {report['best_epoch']}")
    display(pd.DataFrame(report["history"]))
    for partition in ("validation", "test"):
        metrics = report["partitions"][partition]["pytorch"]
        print(f"{model_name}: {partition} errors by expanded rating band")
        display(pd.DataFrame.from_dict(metrics["by_expanded_rating_band"], orient="index"))

        category_metrics = [
            {"rating_band": band, "category": category, **values}
            for band, categories in metrics["by_rating_band_category"].items()
            for category, values in categories.items()
        ]
        print(f"{model_name}: {partition} errors by rating band and category")
        display(pd.DataFrame(category_metrics))

    print(f"{model_name}: test errors by time category")
    display(pd.DataFrame.from_dict(
        report["partitions"]["test"]["pytorch"]["by_category"], orient="index"
    ))

sample = export_sample(training_rows, result.models)
verification = export_bundle(
    result.models,
    directory=settings.artifact_directory,
    sample=sample,
    metadata=result.metadata,
    report=result.report,
    feature_config=training_feature_config,
)

print("Export verification:")
display(pd.Series(verification).to_frame("value"))
print(f"PyTorch model bundle: {settings.artifact_directory.resolve()}")

,model,partition,estimator,rows,players,mae,rmse,player_macro_mae,signed_bias
0,pooled,train,lightgbm,6924,4137,306.671806,406.135408,281.073621,-28.621581
1,pooled,train,ridge,6924,4137,371.532372,458.294649,369.453509,-6.330705
2,pooled,train,train_median,6924,4137,569.578856,687.863983,564.368144,-102.598209
3,pooled,validation,lightgbm,654,385,383.504647,482.727748,376.085343,-85.351044
4,pooled,validation,ridge,654,385,408.787855,500.104248,406.505019,-63.741295
5,pooled,validation,train_median,654,385,606.452599,749.662707,640.352031,-213.963303
6,pooled,test,lightgbm,639,347,395.030936,486.604236,384.458414,-78.133089
7,pooled,test,ridge,639,347,410.168222,499.243496,403.581049,-70.563847
8,pooled,test,train_median,639,347,655.018779,786.680913,646.893295,-221.810642


pooled: validation errors by expanded rating band


,rows,players,mae,rmse,player_macro_mae,signed_bias
<100,0,0,NaN,NaN,NaN,NaN
100-350,54,35,338.478998,415.025605,312.354070,325.959351
350-600,66,42,421.379113,511.725513,398.718403,376.501752
600-850,77,52,319.331139,405.144086,288.072605,228.283120
850-1100,74,48,234.555599,295.906407,223.428871,75.655717
1100-1350,99,41,327.025281,398.361410,332.391773,-98.690318
1350-1600,67,38,337.261933,442.214371,293.966236,-174.828413
1600-1850,29,26,329.676416,390.739439,342.480902,-152.608994
1850-2100,59,38,468.406853,575.656301,478.641102,-436.330643
2100-2350,53,29,542.545764,648.123620,588.609381,-536.228654


pooled: validation errors by rating band and category


,rating_band,category,rows,players,mae,rmse,player_macro_mae,signed_bias
0,<100,blitz,0,0,NaN,NaN,NaN,NaN
1,<100,bullet,0,0,NaN,NaN,NaN,NaN
2,<100,classical,0,0,NaN,NaN,NaN,NaN
3,<100,rapid,0,0,NaN,NaN,NaN,NaN
4,100-350,blitz,34,18,338.432389,419.145276,262.261433,329.170085
5,100-350,bullet,14,11,366.977953,442.234146,416.201211,341.182051
6,100-350,classical,0,0,NaN,NaN,NaN,NaN
7,100-350,rapid,6,6,272.245556,313.605158,272.245556,272.245556
8,350-600,blitz,44,24,460.942104,536.500746,428.716163,427.473956
9,350-600,bullet,5,5,324.192505,352.141465,324.192505,271.550090


pooled: test errors by expanded rating band


... (5 outputs left out)

,gain,gain_fraction,splits
self_phase,13022.142370,0.166233,325
opponent_accuracy,10218.285542,0.130441,331
self_accuracy,9957.391097,0.127110,400
opponent_phase,9878.967349,0.126109,344
self_clock,8994.016104,0.114813,406
opponent_clock,8309.236072,0.106071,345
material,5799.854838,0.074038,223
context,4663.736173,0.059535,210
self_position,2727.049641,0.034812,156
opponent_position,2712.648469,0.034628,122


Export verification:


,value
sample_rows,64.00000
max_absolute_difference,0.00000
warm_single_row_ms,0.43458
native_model_bytes,448618.00000


Model bundle: C:\Users\mc678p\OneDrive - AT&T Services, Inc\Documents\Tower Strategy & Roaming\Analysis\ELO Estimator\artifacts\rating_model_expanded


In [ ]:
predictor = RatingPredictor(settings.artifact_directory)

held_out = training_rows.loc[training_rows["partition"] == "test"].head(
    settings.preview_rows
)

predictions = predictor.predict(
    held_out.loc[:, list(FEATURE_NAMES)],
    categories=held_out["category"].tolist(),
)

prediction_preview = held_out.loc[:, ["target_username", "category", "rating"]].copy()
prediction_preview["predicted_rating"] = predictions
prediction_preview["absolute_error"] = (
    prediction_preview["predicted_rating"] - prediction_preview["rating"]
).abs()

display(prediction_preview)

print("Inference used the exported PyTorch network; no engine analysis was rerun")

,target_username,category,rating,predicted_rating,absolute_error
52,suryaprakash05,bullet,204,356.263158,152.263158
71,yickyb0y,bullet,122,-320.817526,442.817526
102,reiresende,blitz,1148,1807.301760,659.301760
104,reiresende,blitz,1148,633.011811,514.988189
105,reiresende,blitz,1156,1429.791347,273.791347
106,timmyturbo08,blitz,1170,1479.993743,309.993743
109,reiresende,blitz,1180,1235.586122,55.586122
123,gopicheckmates,bullet,519,362.804583,156.195417
150,yuswarditok,blitz,2056,712.901396,1343.098604
180,jcpoza1971,blitz,2509,1962.557872,546.442128


Inference used the exported native model; no engine analysis was rerun


In [13]:
from datetime import UTC, datetime, timedelta

from elo_model.analysis import AnalysisCache, StockfishAnalyzer
from elo_model.game import GameInput

username = "MCodr"
game_index = 0

current_month = datetime.now(UTC).date().replace(day=1)
previous_month = current_month - timedelta(days=1)

recent_settings = replace(
    settings.collection,
    months=(
        (current_month.year, 10),
        (previous_month.year, 9),
    ),
    max_games_per_player=10,
)

recent_games = ModelDatasetBuilder(
    ChessComConnection(recent_settings)
).build([username], target_games=10)

if not 0 <= game_index < len(recent_games):
    raise ValueError(f"Choose an index from {len(recent_games)} eligible games")

selected = recent_games.iloc[game_index+2]
game = GameInput.from_pgn(selected["pgn"])

with StockfishAnalyzer(
    settings.engine,
    AnalysisCache(settings.engine_cache_directory),
) as analyzer:
    _, analysis = analyzer.analyze(game, progress=True)
    estimates = predictor.predict_game(
        game, analysis, engine_metadata=analyzer.metadata
    )

comparison = pd.DataFrame([
    {
        "player": selected[f"{color}_username"],
        "color": color,
        "actual_rating": selected[f"{color}_rating"],
        "estimated_rating": round(estimates[color]),
    }
    for color in ("white", "black")
])
comparison["absolute_error"] = (
    comparison["estimated_rating"] - comparison["actual_rating"]
).abs()

print(game.headers.get("Date"), game.headers.get("Result"))
display(comparison)

2026.10.30 1/2-1/2


,player,color,actual_rating,estimated_rating,absolute_error
0,fjam61,white,1428,1363,65
1,MCodr,black,1444,1253,191
